# 02 · Quantum model (QURE Lab, Role 1)

Four-qubit variational classifier in Qiskit. **Training uses only the train split; restart selection and the decision threshold use only the validation split. The test split is not touched in this notebook.**

Pipeline: four standardized features (from Role 2) → angle encoding `2·arctan(x)` → shallow RY + CX ansatz → measure all 4 qubits → `raw_score = P(odd parity)`.

`raw_score` is measurement-derived. It is **not** a calibrated probability of the label.

In [ ]:
# --- Setup: works in Google Colab and locally -------------------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/beniel292-ops/IBM-bin.git"
BRANCH = "quantum-core"          # change if the team merged to main

if IN_COLAB:
    if not os.path.exists("/content/IBM-bin"):
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/IBM-bin"])
        if r.returncode != 0:
            print("Clone failed (branch not pushed yet?). Upload role1_quantum.zip instead:")
            from google.colab import files
            up = files.upload()
            zipname = next(iter(up))
            subprocess.run(["unzip", "-q", "-o", zipname, "-d", "/content/IBM-bin"])
    os.chdir("/content/IBM-bin")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-quantum.txt", "scikit-learn"])
else:
    root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(root)
sys.path.insert(0, os.getcwd())
import qiskit, qiskit_aer
print("cwd:", os.getcwd(), "| qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)

In [ ]:
# --- Data source ---------------------------------------------------------------------------
# USE_FIXTURE = True  -> SYNTHETIC development fixture (NOT medical data, never evidence)
# USE_FIXTURE = False -> Role 2's real files (input contract: docs/QUANTUM_INPUT_CONTRACT.md)
USE_FIXTURE = False
REAL_FEATURES = "data/lidc/features.csv"
REAL_META = "data/lidc/features_meta.json"

from src.quantum import data_io
if USE_FIXTURE:
    csv_path, meta_path = data_io.make_fixture("results/fixtures/input")
else:
    csv_path, meta_path = REAL_FEATURES, REAL_META
ds = data_io.load_dataset(csv_path, meta_path)      # also checks patient leakage
print("data_status:", ds.meta["data_status"])
print("label definition:", ds.meta["label_definition"])
print("features:", ds.meta["feature_names"])
for s in ("train", "val", "test"):
    idx, X, y = ds.subset(s)
    print(f"{s:5s}: {len(idx):4d} cases, positives {int(y.sum())}, patients {len(set(ds.patient_group_id[i] for i in idx))}")
if ds.meta["data_status"] == "synthetic-fixture":
    print("\n*** SYNTHETIC FIXTURE: results below are for development only ***")

## The circuit
One case, with real angles. `transpiled_ops` are counted in the basis `rz, sx, x, cx` at optimization level 0.

In [ ]:
from src.quantum import model
import numpy as np, json
info = model.circuit_info(layers=2, entangle=True)
print(info["text_drawing"])
print(json.dumps({k: v for k, v in info.items() if k != "text_drawing"}, indent=1))

## Correctness check: fast exact path == Qiskit Statevector
Training uses a fast exact evaluator built from Qiskit's `Operator`. This cell proves it matches the full Qiskit circuit.

In [ ]:
rng = np.random.default_rng(0)
th = rng.uniform(0, 2*np.pi, model.n_params(2))
_, Xva, yva = ds.subset("val")
fast = model.exact_scores(Xva[:10], th, 2, True)
slow = np.array([model.statevector_score(x, th, 2, True) for x in Xva[:10]])
print("max |fast - statevector| =", float(np.max(np.abs(fast - slow))))
assert np.allclose(fast, slow, atol=1e-10)

## Train (noiseless, exact) and freeze
Class-balanced binary cross-entropy, SciPy COBYLA, 3 seeded restarts. The best restart is chosen by **validation** loss, and the threshold by validation balanced accuracy.

In [ ]:
from src.quantum.train import train
_, Xtr, ytr = ds.subset("train")
_, Xva, yva = ds.subset("val")
os.makedirs("results/models", exist_ok=True)
m_ent = train(Xtr, ytr, Xva, yva, layers=2, entangle=True, restarts=3, maxiter=200,
              data_status=ds.meta["data_status"])
p_ent = f"results/models/{m_ent.model_id}-{m_ent.param_version}.json"; m_ent.save(p_ent)
print("saved", p_ent, "| val balanced acc", round(m_ent.val_balanced_accuracy, 3), "| threshold", m_ent.threshold)

## Ablation: same circuit without entangling gates
Same parameters, same observable, no CX gates. We do **not** assume entanglement helps; this cell measures it.

In [ ]:
m_noent = train(Xtr, ytr, Xva, yva, layers=2, entangle=False, restarts=3, maxiter=200,
                data_status=ds.meta["data_status"])
p_noent = f"results/models/{m_noent.model_id}-{m_noent.param_version}.json"; m_noent.save(p_noent)
print("entangled   : val balanced acc", round(m_ent.val_balanced_accuracy, 3))
print("no entangle : val balanced acc", round(m_noent.val_balanced_accuracy, 3))

In [ ]:
import matplotlib.pyplot as plt
s = model.exact_scores(Xva, m_ent.theta, 2, True)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(s[yva == 0], bins=15, alpha=.6, label="label 0")
ax.hist(s[yva == 1], bins=15, alpha=.6, label="label 1")
ax.axvline(m_ent.threshold, color="k", ls="--", label=f"threshold {m_ent.threshold}")
ax.set_xlabel("raw_score (noiseless, validation)"); ax.set_ylabel("cases"); ax.legend()
ax.set_title("Validation raw scores" + (" — SYNTHETIC FIXTURE" if ds.meta["data_status"] != "real" else ""))
plt.tight_layout(); plt.show()

**Next:** notebook 03 loads the frozen model file printed above and runs the noise and mitigation experiments.